# Uneven multimodal transport expansion reshapes regional economic divergence in Africa

**Purpose.** Generate the analysis and figures for uneven multimodal infrastructure expansion and regional economic divergence.

**Inputs.** Country-year transport, GDP, and administrative-boundary data.

**Outputs.** Figure panels and summary statistics for the main economic-divergence analysis.

Run the code cells from top to bottom after updating the local paths described in the repository README. The notebook has no stored outputs; figures and tables are generated when it is executed.

In [ ]:
# Cell | Six-mode transport panel and regional data assembly
import os
from getpass import getpass
from xyzservices import TileProvider


CARTO_API_KEY = os.getenv('CARTO_BASEMAP_KEY')

if not CARTO_API_KEY:
    CARTO_API_KEY = getpass('Please enter CARTO Basemaps API Key:').strip()

if not CARTO_API_KEY:
    raise ValueError('CARTO Basemaps API Key is not provided')

CARTO_POSITRON_NOLABELS = TileProvider(
    name='CARTO.PositronNoLabels.Authenticated',
    url=(
        'https://{s}.basemaps.cartocdn.com/'
        'rastertiles/light_nolabels/{z}/{x}/{y}.png?key={key}'
    ),
    attribution='© OpenStreetMap contributors © CARTO',
    subdomains='abcd',
    max_zoom=20,
    key=CARTO_API_KEY
)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import geopandas as gpd
import contextily as ctx
from matplotlib.colorbar import ColorbarBase
from matplotlib.gridspec import GridSpec
from matplotlib.patches import FancyBboxPatch
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 12
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['figure.dpi'] = 300

# ============================================================================
# Map countries to the five African subregions used throughout the figures.
region_mapping = {
    'North Africa': ['Algeria', 'Egypt', 'Libya', 'Mauritania', 'Morocco', 'Tunisia'],
    'West Africa': ['Benin', 'Burkina Faso', "Ivory Coast", 'Gambia', 'Ghana',
                    'Guinea', 'Guinea-Bissau', 'Liberia', 'Mali', 'Niger',
                    'Nigeria', 'Senegal', 'Sierra Leone', 'Togo'],
    'Central Africa': ['Cameroon', 'Central African Republic', 'Chad', 'Republic of the Congo',
                       'Democratic Republic of the Congo', 'Equatorial Guinea', 'Gabon'],
    'East Africa': ['Burundi', 'Djibouti', 'Eritrea', 'Ethiopia', 'Kenya',
                    'Rwanda', 'Somalia', 'South Sudan', 'Sudan', 'United Republic of Tanzania', 'Uganda'],
    'Southern Africa': ['Angola', 'Botswana', 'eSwatini', 'Lesotho', 'Madagascar', 'Malawi',
                        'Mozambique', 'Namibia', 'South Africa', 'Zambia', 'Zimbabwe']
}
country_to_region = {c: r for r, cs in region_mapping.items() for c in cs}
all_mapped_countries = list(country_to_region.keys())

# ============================================================================
df_transport = pd.read_excel(r'E:\Africa_Infrastructure_Data\transport_statistics_2015_2025.xlsx')
df_transport['region'] = df_transport['country'].map(country_to_region)
df_transport = df_transport[df_transport['country'].isin(all_mapped_countries)].copy()
df_transport = df_transport[(df_transport['year'] >= 2015) & (df_transport['year'] <= 2024)].copy()

print("Data columns:")
print(df_transport.columns.tolist())

road_type_cols = ['road_motorway_km', 'road_trunk_km', 'road_primary_km', 'road_secondary_km', 'road_tertiary_km']
df_transport['total_road_km'] = df_transport[road_type_cols].sum(axis=1)

df_transport['paved_highway_km'] = df_transport['motorway_trunk_paved_km']

station_candidates = [c for c in df_transport.columns if 'station' in c.lower() or 'stop' in c.lower()]

print(f"\nCandidate railway station columns: {station_candidates}")
if station_candidates:
    STATION_COL = station_candidates[0]
    print(f"Using column: {STATION_COL}")
else:
    STATION_COL = 'railway_station_count'
    df_transport[STATION_COL] = 0

    print(f"No railway station column found; creating a placeholder column: {STATION_COL}")
transport_6cols = ['total_road_km', 'paved_highway_km', 'railway_length_km',
                   STATION_COL, 'port_area_km2', 'airport_area_km2']
transport_6labels = {
    'total_road_km':   'Road Length',
    'paved_highway_km':'Paved Highway Length',
    'railway_length_km':'Railway Length',
    STATION_COL:       'Railway Station Count',
    'port_area_km2':   'Port Area',
    'airport_area_km2':'Airport Area',
}

# ============================================================================
df_economic = pd.read_csv(r'E:\Africa_Infrastructure_Data\afdb\ObservationData_fhkvkjf_constant.csv')
country_name_mapping = {
    "Congo, Dem. Rep.": "Democratic Republic of the Congo",
    "Congo, Rep.": "Republic of the Congo",
    "Tanzania": "United Republic of Tanzania",
    "Eswatini": "eSwatini",
    "Cote d'Ivoire": "Ivory Coast"
}
df_economic['country'] = df_economic['country'].replace(country_name_mapping)
df_economic['region'] = df_economic['country'].map(country_to_region)
df_economic = df_economic[df_economic['country'].isin(all_mapped_countries)].copy()
df_econ_filtered = df_economic[(df_economic['Date'] >= 2015) & (df_economic['Date'] <= 2024)].copy()

df_gdp = df_econ_filtered[df_econ_filtered['indicator'] == 'GDP (constant 2000 US$)'][
    ['country', 'Date', 'Value']].rename(columns={'Date': 'year', 'Value': 'GDP_constant_2000_USD'})

# ============================================================================
# Read the spatial boundary layer.
# ============================================================================
gdf = gpd.read_file(r'E:\Africa_Infrastructure_Data\boundaries_project_49\Country_Subregion.shp')
gdf['cx'] = gdf.geometry.centroid.x
gdf['cy'] = gdf.geometry.centroid.y
xmin_g, ymin_g, xmax_g, ymax_g = gdf.total_bounds
map_w = xmax_g - xmin_g
map_h = ymax_g - ymin_g
pad_x   = map_w * 0.04
pad_top = map_h * 0.03
pad_bot = map_h * (-0.06)
VX0, VX1 = xmin_g - pad_x, xmax_g + pad_x
VY0, VY1 = ymin_g - pad_bot, ymax_g + pad_top

gdf_dissolve = gpd.read_file(
    r'E:\Africa_Infrastructure_Data\boundaries_project_49\Country_Subregion_Dissolve.shp')
region_col = gdf_dissolve.columns[0]
_draw_order = ['North Africa', 'East Africa', 'West Africa', 'Southern Africa', 'Central Africa']
_order_map = {r: i for i, r in enumerate(_draw_order)}
gdf_dissolve_sorted = gdf_dissolve.copy()
gdf_dissolve_sorted['_order'] = gdf_dissolve_sorted[region_col].map(_order_map)
gdf_dissolve_sorted = gdf_dissolve_sorted.sort_values('_order').reset_index(drop=True)
gdf_boundaries = gdf_dissolve_sorted.copy()
gdf_boundaries['geometry'] = gdf_boundaries.geometry.boundary

gdf_base = gdf.copy()
gdf_base['region'] = gdf_base['国家'].map(country_to_region)
crs_str = gdf.crs.to_string()

region_border_colors = {
    'Central Africa':  '#0057B8',
    'East Africa':     '#8B00FF',
    'North Africa':    '#00AAAA',
    'Southern Africa': '#CC5500',
    'West Africa':     '#CC0077',
}
BORDER_LW = 2.4

print(f"\n✓ Data loaded: transport {df_transport.shape}, geographic boundaries for {len(gdf)} countries")
print(f"  GDP data: {df_gdp.shape}, years: {sorted(df_gdp['year'].unique())}")
print(f"  transport_6cols: {transport_6cols}")

In [ ]:
# Cell | Cartographic annotation helpers
# ============================================================================
def add_compass(ax, x=0.88, y=0.93, size=0.038):
    from matplotlib.patches import Polygon as P
    ax.add_patch(P(np.array([[x, y+size], [x-size*.4, y], [x+size*.4, y]]),
                   closed=True, fc='#222', ec='white', lw=.5,
                   transform=ax.transAxes, zorder=12))
    ax.add_patch(P(np.array([[x, y-size], [x-size*.4, y], [x+size*.4, y]]),
                   closed=True, fc='white', ec='#222', lw=.5,
                   transform=ax.transAxes, zorder=12))
    ax.text(x, y+size*1.3, 'N', transform=ax.transAxes,
            ha='center', va='center', fontsize=18, fontweight='bold',
            fontfamily='Arial', color='#222', zorder=13)

def add_scalebar(ax, seg_km=750, n_seg=2, map_width_m=None):  
    from matplotlib.patches import Rectangle
    if map_width_m is not None:
        bw = (seg_km * n_seg * 1000) / map_width_m
    else:
        bw = 0.30
    bx, by, bh = 0.73, 0.05, 0.012
    sw = bw / n_seg
    for i in range(n_seg):  # Iterate over the selected analysis inputs.
        ax.add_patch(Rectangle((bx+i*sw, by), sw, bh,
                               fc='#111' if i%2==0 else '#eee',
                               ec='#333', lw=.6,
                               transform=ax.transAxes, zorder=11, clip_on=False))  
    for i, km in enumerate([0, seg_km, seg_km*n_seg]):  # Iterate over the selected analysis inputs.
        ax.text(bx+i*sw, by-.008, f'{km}' if km<seg_km*n_seg else f'{km} km',
                transform=ax.transAxes, ha='center', va='top', fontsize=18,
                fontfamily='Arial', color='#333', zorder=12)  

def draw_country_borders(ax):
    """Draw country borders only."""
    gdf_base.plot(ax=ax, color='none', edgecolor='#666', linewidth=0.8, zorder=5)

print("✓ Helper functions defined")


In [ ]:
# Cell | GDP maps for 2015, 2019 and 2024
# ============================================================================
# Use three benchmark years to show the spatial evolution of GDP and infrastructure.
map_years = [2015, 2019, 2024]
subplot_labels_row1 = ['A', 'B', 'C']

N_CLASS_GDP = 7

gdp_base = df_gdp[df_gdp['year'] == 2015][['country', 'GDP_constant_2000_USD']].set_index('country')

gdp_by_year = {}
gdp_growth_by_year = {}
gdp_add_by_year = {}
for yr in map_years:
    df_yr = df_gdp[df_gdp['year'] == yr][['country', 'GDP_constant_2000_USD']].set_index('country')
    gdp_by_year[yr] = df_yr
    if yr != 2015:
        growth = ((df_yr['GDP_constant_2000_USD'] - gdp_base['GDP_constant_2000_USD'])
                  / gdp_base['GDP_constant_2000_USD'] * 100)
        gdp_add_by_year[yr] = (df_yr['GDP_constant_2000_USD'] - gdp_base['GDP_constant_2000_USD']).reset_index()
        gdp_add_by_year[yr].columns = ['country', 'gdp_growth']
        gdp_growth_by_year[yr] = growth.reset_index()
        gdp_growth_by_year[yr].columns = ['country', 'gdp_growth_rate']

gdp_breaks = [1, 10, 30, 50, 100, 200, 300, 511]
gdp_tick_labels = ['1', '10', '30', '50', '100', '200', '300', '>300']
gdp_cmap = plt.get_cmap('YlOrRd', N_CLASS_GDP)
gdp_norm = mcolors.BoundaryNorm(gdp_breaks, N_CLASS_GDP)

bub_breaks = [-44.2, -20.0, 0.0, 20.0, 40.0, 60.0, 88.1]
N_CLASS_BUB = 6

bub_labels = ['< -20%', '-20–0%', '0–20%', '20–40%', '40–60%', '> 60%']

neg_colors = ['#1a9850', '#66bd63']
pos_colors = ['#abd9e9', '#74add1', '#4575b4', '#053061']
bub_color_map = {0: neg_colors[0], 1: neg_colors[1],
                 2: pos_colors[0], 3: pos_colors[1],
                 4: pos_colors[2], 5: pos_colors[3]}

pos_sizes = [120.0, 230.0, 380.0, 650.0]
bub_sizes = {0: pos_sizes[2], 1: pos_sizes[1],
             2: pos_sizes[0], 3: pos_sizes[1],
             4: pos_sizes[2], 5: pos_sizes[3]}

def get_bub_class(v, brks):
    for i in range(1, len(brks)):
        if v < brks[i]:
            return i - 1
    return len(brks) - 2

data_aspect = (VX1 - VX0) / (VY1 - VY0)
FIG_W = 26
col_w_frac = (0.99 - 0.01 - 2*0.008) / 3
row_h_inch = FIG_W * col_w_frac / data_aspect
FIG_H = row_h_inch + 0.5

fig1, axes1 = plt.subplots(1, 3, figsize=(FIG_W, FIG_H),
                            gridspec_kw={'left': 0.01, 'right': 0.99,
                                         'top': 0.97, 'bottom': 0.03,
                                         'wspace': 0.005})
fig1.patch.set_facecolor('white')

map_data_w = VX1 - VX0
map_data_h = VY1 - VY0

leg_x0_frac = 0.03
leg_w_frac  = 0.32
leg_x0      = VX0 + map_data_w * leg_x0_frac
leg_w       = map_data_w * leg_w_frac
row_spacing = map_data_h * 0.035
title_h     = map_data_h * 0.065
leg_h       = title_h + N_CLASS_BUB * row_spacing + map_data_h * 0.010

cb_x0_ax     = 0.03
cb_y0_ax     = 0.05
cb_w_frac_ax = 0.42
cb_h_frac_ax = 0.030

cb_h_dat = map_data_h * cb_h_frac_ax
cb_gap_v = map_data_h * 0.045
leg_y0   = VY0 + map_data_h * 0.08 + cb_h_dat + cb_gap_v

for col_i, yr in enumerate(map_years):
    ax = axes1[col_i]
    ax.set_xlim(VX0, VX1)
    ax.set_ylim(VY0, VY1)

    gdf_plot = gdf_base.merge(
        gdp_by_year[yr].reset_index().rename(columns={'GDP_constant_2000_USD': 'gdp_val'}),
        left_on='国家', right_on='country', how='left')
    gdf_plot['gdp_val_bn'] = gdf_plot['gdp_val'] / 1e9

    ctx.add_basemap(
        ax,
        crs=crs_str,
        source=CARTO_POSITRON_NOLABELS,
        zoom=4,
        zorder=1,
        attribution='© OpenStreetMap contributors © CARTO',
        attribution_size=6
    )

    gdf_nodata = gdf_plot[gdf_plot['gdp_val_bn'].isna()]
    gdf_data   = gdf_plot[gdf_plot['gdp_val_bn'].notna()]
    if not gdf_nodata.empty:
        gdf_nodata.plot(ax=ax, color='#D0D0D0', edgecolor='none', linewidth=0, zorder=2, alpha=0.88)
    if not gdf_data.empty:
        gdf_data.plot(ax=ax, column='gdp_val_bn', cmap=gdp_cmap, norm=gdp_norm,
                      edgecolor='none', linewidth=0, zorder=2, alpha=0.88)

    if yr in gdp_growth_by_year:
        df_growth = gdp_growth_by_year[yr].merge(
            gdf[['国家', 'cx', 'cy']], left_on='country', right_on='国家', how='left').dropna()
        for cls_i in range(N_CLASS_BUB):
            mask = df_growth['gdp_growth_rate'].apply(
                lambda v: get_bub_class(v, bub_breaks) == cls_i)
            sub = df_growth[mask]
            if sub.empty:
                continue
            ax.scatter(sub['cx'], sub['cy'],
                       s=bub_sizes[cls_i], c=bub_color_map[cls_i],
                       edgecolors='white', linewidths=0.8,
                       zorder=8, alpha=0.92)

    draw_country_borders(ax)
    ax.axis('off')
    for sp in ax.spines.values():
        sp.set_visible(True); sp.set_color('#999'); sp.set_linewidth(0.8)
    ax.set_frame_on(True)

    ax.text(0.015, 0.985, subplot_labels_row1[col_i],
            transform=ax.transAxes, fontsize=35, fontweight='bold',
            va='top', fontfamily='Arial', color='#111', zorder=15)
    ax.text(0.5, 0.985, str(yr), transform=ax.transAxes,
            fontsize=30, ha='center', va='top',
            fontfamily='Arial', color='#111', zorder=15)

    add_compass(ax)
    add_scalebar(ax, map_width_m=(VX1 - VX0))

    # ── GDP colorbar ──────────────────────────────────────────────────────
    cax_gdp = ax.inset_axes([cb_x0_ax, cb_y0_ax, cb_w_frac_ax, cb_h_frac_ax])
    sm_gdp = cm.ScalarMappable(cmap=gdp_cmap, norm=gdp_norm)
    sm_gdp.set_array([])
    cb_gdp = plt.colorbar(sm_gdp, cax=cax_gdp, orientation='horizontal')
    cb_gdp.set_ticks(gdp_breaks)
    cb_gdp.set_ticklabels(gdp_tick_labels)
    cb_gdp.ax.tick_params(labelsize=18, length=3)
    cb_gdp.ax.xaxis.set_ticks_position('bottom')
    for lbl in cb_gdp.ax.get_xticklabels():
        lbl.set_fontfamily('Arial')
    ax.text(cb_x0_ax + cb_w_frac_ax / 2,
            cb_y0_ax + cb_h_frac_ax + 0.013,
            'GDP (Billion USD, constant 2000)',
            transform=ax.transAxes, ha='center', va='bottom',
            fontsize=18, fontfamily='Arial',
            color='#222', zorder=15)

    if yr in gdp_growth_by_year:
        bg = mpatches.FancyBboxPatch(
            (leg_x0, leg_y0), leg_w, leg_h,
            boxstyle='square,pad=0', fc='#f2f2f2', ec='#aaa',
            lw=0.8, alpha=0.92, zorder=14, transform=ax.transData)
        ax.add_patch(bg)

        ax.text(leg_x0 + leg_w * 0.5, leg_y0 + leg_h - map_data_h * 0.005,
                'GDP Growth Rate\n(vs 2015)',
                ha='center', va='top', fontsize=18,
                fontfamily='Arial', color='#222',
                transform=ax.transData, zorder=15, linespacing=1.2)

        circ_x = leg_x0 + leg_w * 0.13
        text_x = leg_x0 + leg_w * 0.27
        for i in range(N_CLASS_BUB):
            ry = leg_y0 + leg_h - title_h - (i + 0.5) * row_spacing
            ax.scatter(circ_x, ry, s=bub_sizes[i],
                       c=bub_color_map[i], edgecolors='white', linewidths=0.8,
                       zorder=15, alpha=0.95)
            ax.text(text_x, ry, bub_labels[i],
                    ha='left', va='center', fontsize=18,
                    fontfamily='Arial', color='#333',
                    transform=ax.transData, zorder=15)

plt.show()
print("✓ Row 1 GDP distribution maps completed")


In [ ]:
# Cell | GDP growth-data inspection
df_gdp_add = pd.DataFrame(gdp_add_by_year[2024])
df_gdp_add['gdp_growth'] = (df_gdp_add['gdp_growth'] / 1e9).round(2)
df_gdp_add = df_gdp_add.sort_values(by='country')
df_gdp_add = df_gdp_add.reset_index(drop=True)
df_gdp_add

In [ ]:
# Cell | GDP growth-table inspection
pd.DataFrame(gdp_growth_by_year[2024]).sort_values(by='country').round(1)

In [ ]:
# Cell | Infrastructure-growth preprocessing and spatial export

import os

# Store reproducible intermediate spatial data and figure inputs in the analysis output directory.
OUT_DIR = r'outputs\multimodal_transport_economic_divergence'
os.makedirs(OUT_DIR, exist_ok=True)

# Read projected infrastructure layers for each benchmark year.
road_shp_template    = r'E:\Africa_Infrastructure_Data\infrastructure\clip_project\{year}\africa_major_roads_line.shp'
railway_shp_template = r'E:\Africa_Infrastructure_Data\infrastructure\clip_project\{year}\africa_railways_line.shp'
station_shp_template = r'E:\Africa_Infrastructure_Data\infrastructure\clip_project\{year}\africa_railways_polygon.shp'
port_shp_template    = r'E:\Africa_Infrastructure_Data\infrastructure\clip_project\{year}\africa_ports_polygon.shp'
airport_shp_template = r'E:\Africa_Infrastructure_Data\infrastructure\clip_project\{year}\africa_airports_polygon.shp'

paved_surfaces = [
    'paved', 'asphalt', 'chipseal', 'concrete', 'concrete:lanes', 
    'concrete:plates', 'paving_stones', 'paving_stones:lanes', 
    'grass_paver', 'sett', 'unhewn_cobblestone', 'cobblestone', 
    'bricks', 'metal', 'metal_grid', 'wood', 'stepping_stones', 
    'rubber', 'tiles', 'fibre_reinforced_polymer_grate'
]

AREA_THRESHOLD_KM2 = 1.0
target_crs = gdf.crs

def save_shp(gdf_in, path):
    gdf_out = gdf_in.copy()
    # Work on a copy to preserve the source table.
    gdf_out = gdf_out[~gdf_out.geometry.isna()].copy()
    gdf_out = gdf_out[gdf_out.geometry.is_valid].copy()
    
    if gdf_out.empty:
        print(f"    Warning: {path} contains no valid geometries; skipping export")
        return
    
    geom_col = gdf_out.geometry.name
    for col in gdf_out.columns:
        if col == geom_col:
            continue
        if not pd.api.types.is_numeric_dtype(gdf_out[col]):
            gdf_out[col] = gdf_out[col].astype(str).where(gdf_out[col].notna(), other=None)
            gdf_out[col] = gdf_out[col].astype(object)
    gdf_out.to_file(path)

def load_point_layer(shp_path, target_crs):
    if not os.path.exists(shp_path):
        return None
    gdf_poly = gpd.read_file(shp_path)
    if gdf_poly.crs != target_crs:
        gdf_poly = gdf_poly.to_crs(target_crs)
    # Work on a copy to preserve the source table.
    gdf_poly = gdf_poly[~gdf_poly.geometry.isna()].copy()
    gdf_poly = gdf_poly[gdf_poly.geometry.is_valid].copy()
    
    if gdf_poly.empty:
        return gdf_poly
    
    gdf_ea = gdf_poly.to_crs('EPSG:6933')
    gdf_poly['area_km2'] = gdf_ea.geometry.area / 1e6
    gdf_poly = gdf_poly[gdf_poly['area_km2'] > AREA_THRESHOLD_KM2].copy()
    if gdf_poly.empty:
        return gdf_poly
    gdf_poly['cx'] = gdf_poly.geometry.centroid.x
    gdf_poly['cy'] = gdf_poly.geometry.centroid.y
    return gdf_poly

def load_station_layer(shp_path, target_crs):
    if not os.path.exists(shp_path):
        return None
    gdf_poly = gpd.read_file(shp_path)
    if gdf_poly.crs != target_crs:
        gdf_poly = gdf_poly.to_crs(target_crs)
    # Work on a copy to preserve the source table.
    gdf_poly = gdf_poly[~gdf_poly.geometry.isna()].copy()
    gdf_poly = gdf_poly[gdf_poly.geometry.is_valid].copy()
    
    if gdf_poly.empty:
        return gdf_poly
    
    gdf_poly['cx'] = gdf_poly.geometry.centroid.x
    gdf_poly['cy'] = gdf_poly.geometry.centroid.y
    return gdf_poly

def load_paved_highway_layer(shp_path, target_crs):
    if not os.path.exists(shp_path):
        return None
    gdf_roads = gpd.read_file(shp_path)
    if gdf_roads.crs != target_crs:
        gdf_roads = gdf_roads.to_crs(target_crs)
    
    # Work on a copy to preserve the source table.
    gdf_roads = gdf_roads[~gdf_roads.geometry.isna()].copy()
    gdf_roads = gdf_roads[gdf_roads.geometry.is_valid].copy()
    
    if gdf_roads.empty:
        return None
    
    # Work on a copy to preserve the source table.
    gdf_mt = gdf_roads[gdf_roads['type'].isin(['motorway', 'trunk'])].copy()
    
    if gdf_mt.empty:
        return None
    
    if 'surface' in gdf_mt.columns:
        gdf_paved = gdf_mt[gdf_mt['surface'].isin(paved_surfaces)].copy()
    else:
        gdf_paved = gdf_mt
    
    return gdf_paved if not gdf_paved.empty else None

def _split_by_buffer_intersect(gdf_yr, gdf_2015, buffer_m):
    if gdf_yr is None or gdf_yr.empty:
        return None, None
    if gdf_2015 is None or gdf_2015.empty:
        return gdf_yr, gpd.GeoDataFrame()
    
    # Work on a copy to preserve the source table.
    gdf_yr = gdf_yr[~gdf_yr.geometry.isna()].copy()
    gdf_yr = gdf_yr[gdf_yr.geometry.is_valid].copy()
    
    if gdf_yr.empty:
        return None, None
    
    gdf_yr_ea   = gdf_yr.to_crs('EPSG:6933').copy().reset_index(drop=True)
    gdf_2015_ea = gdf_2015.to_crs('EPSG:6933').copy().reset_index(drop=True)
    gdf_yr_ea['geometry']   = gdf_yr_ea.geometry.buffer(buffer_m)
    gdf_2015_ea['geometry'] = gdf_2015_ea.geometry.buffer(buffer_m)
    joined = gpd.sjoin(
        gdf_yr_ea[['geometry']],
        gdf_2015_ea[['geometry']],
        how='left', predicate='intersects'
    )
    matched_idx = joined.dropna(subset=['index_right']).index.unique()
    all_idx = gdf_yr_ea.index
    is_unchanged = all_idx.isin(matched_idx)
    return gdf_yr[~is_unchanged].copy(), gdf_yr[is_unchanged].copy()

def split_new_unchanged(gdf_yr, gdf_2015, buffer_m=100):
    return _split_by_buffer_intersect(gdf_yr, gdf_2015, buffer_m)

def split_line_new_unchanged(gdf_yr, gdf_2015, buffer_m=100):
    return _split_by_buffer_intersect(gdf_yr, gdf_2015, buffer_m)

trans_base_country = df_transport[df_transport['year'] == 2015][
    ['country'] + transport_6cols].set_index('country')

trans_growth_by_year = {}
for yr in [2019, 2024]:
    df_yr = df_transport[df_transport['year'] == yr][
        ['country'] + transport_6cols].set_index('country')
    growth = pd.DataFrame(index=trans_base_country.index)
    for col in transport_6cols:
        base_vals = trans_base_country[col]
        yr_vals   = df_yr[col]
        diff      = yr_vals - base_vals
        pct        = diff / base_vals.replace(0, np.nan) * 100
        abs_growth = yr_vals
        growth[col + '_growth'] = pct.where(base_vals != 0, np.nan)
    trans_growth_by_year[yr] = growth.reset_index()
    trans_growth_by_year[yr].to_csv(
        os.path.join(OUT_DIR, f'trans_growth_{yr}.csv'), index=False)

# ==============================================================================
    print(f"  ✓ Saved trans_growth_{yr}.csv")
print("\n── Calculating and exporting absolute transport infrastructure growth and growth rates for 2015-2024 ──")
df_2024 = df_transport[df_transport['year'] == 2024][['country'] + transport_6cols].set_index('country')

growth_24_vs_15 = pd.DataFrame(index=trans_base_country.index)

for col in transport_6cols:
    base_15_vals = trans_base_country[col]
    yr_24_vals   = df_2024[col]
    
    abs_diff = yr_24_vals - base_15_vals
    growth_24_vs_15[f'{col}_abs_growth'] = abs_diff.round(2)
    
    pct_growth = (abs_diff / base_15_vals.replace(0, np.nan)) * 100
    growth_24_vs_15[f'{col}_pct_growth(%)'] = pct_growth.round(1)

# Export results.
output_path_24vs15 = os.path.join(OUT_DIR, 'trans_growth_absolute_and_rate_2024_vs_2015.csv')
growth_24_vs_15.reset_index().to_csv(output_path_24vs15, index=False)
# ==============================================================================

print(f"  ✓ Saved absolute growth and growth rates for 2024 vs 2015: trans_growth_absolute_and_rate_2024_vs_2015.csv")
petal_max_by_col = {}
print("\nPetal chart normalization parameters by infrastructure type (boxplot upper whiskers):")
for col in transport_6cols:
    all_growth_pos = []
    for yr in [2019, 2024]:
        vals = trans_growth_by_year[yr][col + '_growth'].dropna().values
        all_growth_pos.extend(vals[vals > 0].tolist())
    
    if all_growth_pos:
        arr = np.array(all_growth_pos)
        q1, q3 = np.percentile(arr, 25), np.percentile(arr, 75)
        iqr = q3 - q1
        upper_fence = q3 + 1.5 * iqr
        valid = arr[arr <= upper_fence]
        max_val = valid.max() if len(valid) > 0 else upper_fence
    else:
        max_val = 100.0
    
    petal_max_by_col[col] = max_val

    print(f"  {col}: maximum={max_val:.2f}")
pd.DataFrame([petal_max_by_col]).to_csv(
    os.path.join(OUT_DIR, 'petal_params.csv'), index=False)

# Use three benchmark years to show the spatial evolution of GDP and infrastructure.
print(f"  ✓ Saved petal_params.csv")
map_years = [2015, 2019, 2024]

gdf_port_2015    = load_point_layer(port_shp_template.format(year=2015),    target_crs)
gdf_airport_2015 = load_point_layer(airport_shp_template.format(year=2015), target_crs)
gdf_station_2015 = load_station_layer(station_shp_template.format(year=2015), target_crs)

gdf_road_2015, gdf_highway_2015, gdf_rail_2015 = None, None, None
_road_path_2015 = road_shp_template.format(year=2015)
_rail_path_2015 = railway_shp_template.format(year=2015)

if os.path.exists(_road_path_2015):
    gdf_road_2015 = gpd.read_file(_road_path_2015)
    if gdf_road_2015.crs != target_crs:
        gdf_road_2015 = gdf_road_2015.to_crs(target_crs)
    # Work on a copy to preserve the source table.
    gdf_road_2015 = gdf_road_2015[~gdf_road_2015.geometry.isna()].copy()
    gdf_road_2015 = gdf_road_2015[gdf_road_2015.geometry.is_valid].copy()
    gdf_highway_2015 = load_paved_highway_layer(_road_path_2015, target_crs)

if os.path.exists(_rail_path_2015):
    gdf_rail_2015 = gpd.read_file(_rail_path_2015)
    if gdf_rail_2015.crs != target_crs:
        gdf_rail_2015 = gdf_rail_2015.to_crs(target_crs)
    # Work on a copy to preserve the source table.
    gdf_rail_2015 = gdf_rail_2015[~gdf_rail_2015.geometry.isna()].copy()
    gdf_rail_2015 = gdf_rail_2015[gdf_rail_2015.geometry.is_valid].copy()

for yr in map_years:
    yr_dir = os.path.join(OUT_DIR, str(yr))
    os.makedirs(yr_dir, exist_ok=True)

    road_path = road_shp_template.format(year=yr)
    if os.path.exists(road_path):
        gdf_road = gpd.read_file(road_path)
        if gdf_road.crs != target_crs:
            gdf_road = gdf_road.to_crs(target_crs)
        # Work on a copy to preserve the source table.
        gdf_road = gdf_road[~gdf_road.geometry.isna()].copy()
        gdf_road = gdf_road[gdf_road.geometry.is_valid].copy()
        
        if not gdf_road.empty:
            if yr == 2015:
                save_shp(gdf_road, os.path.join(yr_dir, 'road.shp'))
            else:
                gdf_road_new, gdf_road_unch = split_line_new_unchanged(gdf_road, gdf_road_2015)
                if gdf_road_new is not None and not gdf_road_new.empty:
                    save_shp(gdf_road_new, os.path.join(yr_dir, 'road_new.shp'))
                if gdf_road_unch is not None and not gdf_road_unch.empty:
                    save_shp(gdf_road_unch, os.path.join(yr_dir, 'road_unch.shp'))

            print(f"  ✓ {yr} road data saved")
            gdf_highway = load_paved_highway_layer(road_path, target_crs)
            if gdf_highway is not None:
                if yr == 2015:
                    save_shp(gdf_highway, os.path.join(yr_dir, 'highway.shp'))
                else:
                    gdf_highway_new, gdf_highway_unch = split_line_new_unchanged(gdf_highway, gdf_highway_2015)
                    if gdf_highway_new is not None and not gdf_highway_new.empty:
                        save_shp(gdf_highway_new, os.path.join(yr_dir, 'highway_new.shp'))
                    if gdf_highway_unch is not None and not gdf_highway_unch.empty:
                        save_shp(gdf_highway_unch, os.path.join(yr_dir, 'highway_unch.shp'))

                print(f"  ✓ {yr} paved highway data saved")
    railway_path = railway_shp_template.format(year=yr)
    if os.path.exists(railway_path):
        gdf_rail = gpd.read_file(railway_path)
        if gdf_rail.crs != target_crs:
            gdf_rail = gdf_rail.to_crs(target_crs)
        # Work on a copy to preserve the source table.
        gdf_rail = gdf_rail[~gdf_rail.geometry.isna()].copy()
        gdf_rail = gdf_rail[gdf_rail.geometry.is_valid].copy()
        
        if not gdf_rail.empty:
            if yr == 2015:
                save_shp(gdf_rail, os.path.join(yr_dir, 'railway.shp'))
            else:
                gdf_rail_new, gdf_rail_unch = split_line_new_unchanged(gdf_rail, gdf_rail_2015)
                if gdf_rail_new is not None and not gdf_rail_new.empty:
                    save_shp(gdf_rail_new, os.path.join(yr_dir, 'railway_new.shp'))
                if gdf_rail_unch is not None and not gdf_rail_unch.empty:
                    save_shp(gdf_rail_unch, os.path.join(yr_dir, 'railway_unch.shp'))

            print(f"  ✓ {yr} railway data saved")
    gdf_station_yr = load_station_layer(station_shp_template.format(year=yr), target_crs)
    if gdf_station_yr is not None and not gdf_station_yr.empty:
        if yr == 2015:
            gdf_station_yr[['cx', 'cy']].to_csv(
                os.path.join(yr_dir, 'station.csv'), index=False)
        else:
            gdf_new, gdf_unch = split_new_unchanged(gdf_station_yr, gdf_station_2015)
            if gdf_new is not None and not gdf_new.empty:
                gdf_new[['cx', 'cy']].to_csv(
                    os.path.join(yr_dir, 'station_new.csv'), index=False)
            if gdf_unch is not None and not gdf_unch.empty:
                gdf_unch[['cx', 'cy']].to_csv(
                    os.path.join(yr_dir, 'station_unch.csv'), index=False)

        print(f"  ✓ {yr} railway station data saved")
    gdf_port_yr = load_point_layer(port_shp_template.format(year=yr), target_crs)
    if gdf_port_yr is not None and not gdf_port_yr.empty:
        if yr == 2015:
            gdf_port_yr[['cx', 'cy', 'area_km2']].to_csv(
                os.path.join(yr_dir, 'port.csv'), index=False)
        else:
            gdf_new, gdf_unch = split_new_unchanged(gdf_port_yr, gdf_port_2015)
            if gdf_new is not None and not gdf_new.empty:
                gdf_new[['cx', 'cy', 'area_km2']].to_csv(
                    os.path.join(yr_dir, 'port_new.csv'), index=False)
            if gdf_unch is not None and not gdf_unch.empty:
                gdf_unch[['cx', 'cy', 'area_km2']].to_csv(
                    os.path.join(yr_dir, 'port_unch.csv'), index=False)

        print(f"  ✓ {yr} port data saved")
    gdf_airport_yr = load_point_layer(airport_shp_template.format(year=yr), target_crs)
    if gdf_airport_yr is not None and not gdf_airport_yr.empty:
        if yr == 2015:
            gdf_airport_yr[['cx', 'cy', 'area_km2']].to_csv(
                os.path.join(yr_dir, 'airport.csv'), index=False)
        else:
            gdf_new, gdf_unch = split_new_unchanged(gdf_airport_yr, gdf_airport_2015)
            if gdf_new is not None and not gdf_new.empty:
                gdf_new[['cx', 'cy', 'area_km2']].to_csv(
                    os.path.join(yr_dir, 'airport_new.csv'), index=False)
            if gdf_unch is not None and not gdf_unch.empty:
                gdf_unch[['cx', 'cy', 'area_km2']].to_csv(
                    os.path.join(yr_dir, 'airport_unch.csv'), index=False)

        print(f"  ✓ {yr} airport data saved")
print(f"\n✓ Row 2 preprocessing completed; all files saved to:\n  {OUT_DIR}")


In [ ]:
# Cell | Multimodal infrastructure maps for three benchmark years

import os
# Store reproducible intermediate spatial data and figure inputs in the analysis output directory.
OUT_DIR = r'outputs\multimodal_transport_economic_divergence'

# Assign panel labels for the multimodal infrastructure maps.
subplot_labels_row2 = ['D', 'E', 'F']

petal_colors_base = {
    'total_road_km':    '#FF6600',
    'paved_highway_km': '#00008B',
    'railway_length_km':'#00CED1',
    'port_area_km2':    '#C71585',
    'airport_area_km2': '#7B2FBE',
}
# Work on a copy to preserve the source table.
petal_colors = petal_colors_base.copy()
petal_colors[STATION_COL] = '#8B0000'

petal_color_list = [petal_colors[c] for c in transport_6cols]

transport_6labels_vis_base = {
    'total_road_km':    'Road',
    'paved_highway_km': 'Paved Highway',
    'railway_length_km':'Railway',
    'port_area_km2':    'Port',
    'airport_area_km2': 'Aviation',
}
transport_6labels_vis = transport_6labels_vis_base.copy()
transport_6labels_vis[STATION_COL] = 'Railway Station'

trans_growth_by_year = {}
for yr in [2019, 2024]:
    trans_growth_by_year[yr] = pd.read_csv(os.path.join(OUT_DIR, f'trans_growth_{yr}.csv'))

petal_max_by_col = pd.read_csv(os.path.join(OUT_DIR, 'petal_params.csv')).iloc[0].to_dict()

map_data_w = VX1 - VX0
map_data_h = VY1 - VY0
petal_radius_scale = map_data_h * 0.025 * 1.2

MANUAL_OFFSETS = {
    'Gambia':            (-0.08,  0.05,  0.35),
    'Guinea-Bissau':     (-0.07, -0.01, -0.30),
    'Sierra Leone':      (-0.07, -0.04,  0.30),
    'Liberia':           (-0.06, -0.07, -0.25),
    'Togo':              ( 0.03, -0.07, -0.25),
    'Equatorial Guinea': (-0.05, -0.03,  0.30),
    'Burundi':           ( 0.20, -0.05, -0.35),
    'Mali':              ( 0.00,  0.06,  0.0),
    'Cameroon':          ( 0.01,  0.04,  0.0),
    'Ghana':             ( 0.00, -0.02,  0.0),
    'South Africa':      (-0.04, -0.04,  0.0),
    'Uganda':            ( 0.015,  0.025,  0.0),
    'Tunisia':           ( 0.00, -0.02,  0.0),
    'Republic of the Congo': ( 0.02, 0.02,  0.0)
}
NO_ARROW_COUNTRIES = {'Mali', 'Cameroon', 'Ghana', 'South Africa', 'Uganda', 'Tunisia', 'Republic of the Congo'}

PORT_MARKER        = '$⚓$'
AIRPORT_MARKER     = '$✈$'
STATION_MARKER     = '$★$'
PORT_COLOR         = '#005936'
PORT_EDGE          = '#1a7a00'
AIRPORT_COLOR      = '#b8860b'
AIRPORT_EDGE       = '#7a5a00'
STATION_COLOR      = '#8B0000'
STATION_EDGE       = '#660000'
ROAD_COLOR         = '#2166ac'
HIGHWAY_COLOR      = '#00008B'
RAILWAY_COLOR      = '#d6604d'
UNCHANGED_COLOR    = '#999999'
UNCHANGED_EDGE     = '#666666'
POINT_SIZE = 60

# ============================================================================
def draw_petal_chart_v2(ax, cx, cy, values, colors, radius_scale, max_by_col, col_names):
    bg = plt.Circle((cx, cy), radius_scale, color='#444444', alpha=0.35, zorder=8)
    ax.add_patch(bg)
    n = len(values)
    angles = np.linspace(np.pi/2, np.pi/2 + 2*np.pi, n, endpoint=False)
    half_angle = np.pi / n
    for val, color, angle, col_name in zip(values, colors, angles, col_names):
        if np.isnan(val) or val == 0:
            continue
        col_max = max_by_col[col_name]
        r = radius_scale * (min(abs(val), col_max) / col_max) ** 0.5
        r = max(r, radius_scale * 0.05)
        theta = np.linspace(angle - half_angle, angle + half_angle, 30)
        petal_x = cx + r * np.cos(theta)
        petal_y = cy + r * np.sin(theta)
        verts_x = np.concatenate([[cx], petal_x, [cx]])
        verts_y = np.concatenate([[cy], petal_y, [cy]])
        if val > 0:
            ax.fill(verts_x, verts_y, color=color, alpha=0.82, zorder=9)
            ax.plot(verts_x, verts_y, color='white', lw=0.5, zorder=10)
        else:
            ax.fill(verts_x, verts_y, color=color, alpha=0.45, zorder=9)
            ax.plot(verts_x, verts_y, color='white', lw=0.5, zorder=10)
            n_lines = 6
            for k in range(n_lines):
                t = (k + 0.5) / n_lines
                r_line = r * t
                lx = cx + r_line * np.cos(theta)
                ly = cy + r_line * np.sin(theta)
                ax.plot(lx, ly, color='white', lw=1, linestyle='solid',
                        alpha=0.85, zorder=10)

def read_point_csv(path):
    if os.path.exists(path):
        return pd.read_csv(path)
    return pd.DataFrame(columns=['cx', 'cy'])

def read_shp(path):
    if os.path.exists(path):
        return gpd.read_file(path)
    return None

data_aspect = (VX1 - VX0) / (VY1 - VY0)
FIG_W = 26
col_w_frac = (0.99 - 0.01 - 2*0.008) / 3
row_h_inch = FIG_W * col_w_frac / data_aspect
FIG_H = row_h_inch + 0.5

fig2, axes2 = plt.subplots(1, 3, figsize=(FIG_W, FIG_H),
                            gridspec_kw={'left': 0.01, 'right': 0.99,
                                         'top': 0.97, 'bottom': 0.03,
                                         'wspace': 0.005})
fig2.patch.set_facecolor('white')

map_data_w2 = VX1 - VX0
map_data_h2 = VY1 - VY0
leg2_x0_frac = 0.02
leg2_w_frac  = 0.37
leg2_x0      = VX0 + map_data_w2 * leg2_x0_frac
leg2_w       = map_data_w2 * leg2_w_frac
row_sp2      = map_data_h2 * 0.022
title_h2     = map_data_h2 * 0.040
leg_gap      = map_data_h2 * 0.012

for col_i, yr in enumerate(map_years):
    ax = axes2[col_i]
    ax.set_xlim(VX0, VX1)
    ax.set_ylim(VY0, VY1)
    yr_dir = os.path.join(OUT_DIR, str(yr))

    ctx.add_basemap(ax, crs=crs_str,
                    source=CARTO_POSITRON_NOLABELS,
                    zoom=4, zorder=1,
                    attribution='© OpenStreetMap contributors © CARTO', attribution_size=6)
    gdf_base.plot(ax=ax, color='#f5f5f5', edgecolor='none', linewidth=0, zorder=2, alpha=0.7)

    if yr == 2015:
        gdf_road = read_shp(os.path.join(yr_dir, 'road.shp'))
        if gdf_road is not None:
            gdf_road.plot(ax=ax, color=ROAD_COLOR, linewidth=0.35, alpha=0.65, zorder=3)
    else:
        gdf_road_unch = read_shp(os.path.join(yr_dir, 'road_unch.shp'))
        gdf_road_new  = read_shp(os.path.join(yr_dir, 'road_new.shp'))
        if gdf_road_unch is not None:
            gdf_road_unch.plot(ax=ax, color=UNCHANGED_COLOR, linewidth=0.35, alpha=0.55, zorder=3)
        if gdf_road_new is not None:
            gdf_road_new.plot(ax=ax, color=ROAD_COLOR, linewidth=0.35, alpha=0.65, zorder=3)

    if yr == 2015:
        gdf_highway = read_shp(os.path.join(yr_dir, 'highway.shp'))
        if gdf_highway is not None:
            gdf_highway.plot(ax=ax, color=HIGHWAY_COLOR, linewidth=0.6, alpha=0.75, zorder=3.5)
    else:
        gdf_highway_unch = read_shp(os.path.join(yr_dir, 'highway_unch.shp'))
        gdf_highway_new  = read_shp(os.path.join(yr_dir, 'highway_new.shp'))
        if gdf_highway_unch is not None:
            gdf_highway_unch.plot(ax=ax, color=UNCHANGED_COLOR, linewidth=0.6, alpha=0.55, zorder=3.5)
        if gdf_highway_new is not None:
            gdf_highway_new.plot(ax=ax, color=HIGHWAY_COLOR, linewidth=0.6, alpha=0.75, zorder=3.5)

    if yr == 2015:
        gdf_rail = read_shp(os.path.join(yr_dir, 'railway.shp'))
        if gdf_rail is not None:
            gdf_rail.plot(ax=ax, color=RAILWAY_COLOR, linewidth=0.5, alpha=0.75, zorder=4)
    else:
        gdf_rail_unch = read_shp(os.path.join(yr_dir, 'railway_unch.shp'))
        gdf_rail_new  = read_shp(os.path.join(yr_dir, 'railway_new.shp'))
        if gdf_rail_unch is not None:
            gdf_rail_unch.plot(ax=ax, color=UNCHANGED_COLOR, linewidth=0.5, alpha=0.55, zorder=4)
        if gdf_rail_new is not None:
            gdf_rail_new.plot(ax=ax, color=RAILWAY_COLOR, linewidth=0.5, alpha=0.75, zorder=4)

    if yr == 2015:
        df_station = read_point_csv(os.path.join(yr_dir, 'station.csv'))
        if not df_station.empty:
            ax.scatter(df_station['cx'], df_station['cy'],
                       s=POINT_SIZE*0.3, c=STATION_COLOR, marker=STATION_MARKER,
                       edgecolors=STATION_EDGE, linewidths=0.3, zorder=6.5, alpha=0.9)
    else:
        df_station_unch = read_point_csv(os.path.join(yr_dir, 'station_unch.csv'))
        df_station_new  = read_point_csv(os.path.join(yr_dir, 'station_new.csv'))
        if not df_station_unch.empty:
            ax.scatter(df_station_unch['cx'], df_station_unch['cy'],
                       s=POINT_SIZE*0.3, c=UNCHANGED_COLOR, marker=STATION_MARKER,
                       edgecolors=UNCHANGED_EDGE, linewidths=0.3, zorder=6.5, alpha=0.75)
        if not df_station_new.empty:
            ax.scatter(df_station_new['cx'], df_station_new['cy'],
                       s=POINT_SIZE*0.3, c=STATION_COLOR, marker=STATION_MARKER,
                       edgecolors=STATION_EDGE, linewidths=0.3, zorder=6.5, alpha=0.9)

    if yr == 2015:
        df_port = read_point_csv(os.path.join(yr_dir, 'port.csv'))
        if not df_port.empty:
            ax.scatter(df_port['cx'], df_port['cy'],
                       s=POINT_SIZE+10, c=PORT_COLOR, marker=PORT_MARKER,
                       edgecolors=PORT_EDGE, linewidths=0.3, zorder=7, alpha=0.9)
    else:
        df_port_unch = read_point_csv(os.path.join(yr_dir, 'port_unch.csv'))
        df_port_new  = read_point_csv(os.path.join(yr_dir, 'port_new.csv'))
        if not df_port_unch.empty:
            ax.scatter(df_port_unch['cx'], df_port_unch['cy'],
                       s=POINT_SIZE+10, c=UNCHANGED_COLOR, marker=PORT_MARKER,
                       edgecolors=UNCHANGED_EDGE, linewidths=0.3, zorder=7, alpha=0.75)
        if not df_port_new.empty:
            ax.scatter(df_port_new['cx'], df_port_new['cy'],
                       s=POINT_SIZE+10, c=PORT_COLOR, marker=PORT_MARKER,
                       edgecolors=PORT_EDGE, linewidths=0.3, zorder=7, alpha=0.9)

    if yr == 2015:
        df_airport = read_point_csv(os.path.join(yr_dir, 'airport.csv'))
        if not df_airport.empty:
            ax.scatter(df_airport['cx'], df_airport['cy'],
                       s=POINT_SIZE, c=AIRPORT_COLOR, marker=AIRPORT_MARKER,
                       edgecolors=AIRPORT_EDGE, linewidths=0.3, zorder=7, alpha=0.9)
    else:
        df_airport_unch = read_point_csv(os.path.join(yr_dir, 'airport_unch.csv'))
        df_airport_new  = read_point_csv(os.path.join(yr_dir, 'airport_new.csv'))
        if not df_airport_unch.empty:
            ax.scatter(df_airport_unch['cx'], df_airport_unch['cy'],
                       s=POINT_SIZE, c=UNCHANGED_COLOR, marker=AIRPORT_MARKER,
                       edgecolors=UNCHANGED_EDGE, linewidths=0.3, zorder=7, alpha=0.75)
        if not df_airport_new.empty:
            ax.scatter(df_airport_new['cx'], df_airport_new['cy'],
                       s=POINT_SIZE, c=AIRPORT_COLOR, marker=AIRPORT_MARKER,
                       edgecolors=AIRPORT_EDGE, linewidths=0.3, zorder=7, alpha=0.9)

    if yr in trans_growth_by_year:
        df_petal = trans_growth_by_year[yr].merge(
            gdf[['国家', 'cx', 'cy']], left_on='country', right_on='国家', how='left'
        ).dropna(subset=['cx', 'cy'])
        for _, row in df_petal.iterrows():
            country = row['国家']
            cx_orig = row['cx']
            cy_orig = row['cy']
            vals = [row.get(col + '_growth', np.nan) for col in transport_6cols]

            if country in MANUAL_OFFSETS:
                dx_frac, dy_frac, arrow_rad = MANUAL_OFFSETS[country]
                pcx = cx_orig + dx_frac * map_data_w2
                pcy = cy_orig + dy_frac * map_data_h2
                pcx = np.clip(pcx, VX0 + petal_radius_scale, VX1 - petal_radius_scale)
                pcy = np.clip(pcy, VY0 + petal_radius_scale, VY1 - petal_radius_scale)
                draw_petal_chart_v2(ax, pcx, pcy, vals,
                                    petal_color_list, petal_radius_scale, petal_max_by_col, transport_6cols)
                if country not in NO_ARROW_COUNTRIES:
                    ax.annotate('',
                        xy=(cx_orig, cy_orig), xytext=(pcx, pcy),
                        arrowprops=dict(
                            arrowstyle='->', color='#333333', lw=1.2,
                            connectionstyle=f'arc3,rad={arrow_rad}',
                            mutation_scale=12,
                        ),
                        zorder=11)
            else:
                draw_petal_chart_v2(ax, cx_orig, cy_orig, vals,
                                    petal_color_list, petal_radius_scale, petal_max_by_col, transport_6cols)

    draw_country_borders(ax)
    ax.axis('off')
    for sp in ax.spines.values():
        sp.set_visible(True); sp.set_color('#999'); sp.set_linewidth(0.8)
    ax.set_frame_on(True)

    ax.text(0.015, 0.985, subplot_labels_row2[col_i],
            transform=ax.transAxes, fontsize=35, fontweight='bold',
            va='top', fontfamily='Arial', color='#111', zorder=15)
    ax.text(0.5, 0.985, str(yr), transform=ax.transAxes,
            fontsize=30, ha='center', va='top',
            fontfamily='Arial', color='#111', zorder=15)

    add_compass(ax)
    add_scalebar(ax, map_width_m=(VX1 - VX0))

    sym_x  = leg2_x0 + leg2_w * 0.08
    text_x = leg2_x0 + leg2_w * 0.20

    if yr == 2015:
        leg_items = [
            ('line',  ROAD_COLOR,     None,          1.5, 0.65, 'Road'),
            ('line',  HIGHWAY_COLOR,  None,          2.0, 0.75, 'Paved Highway'),
            ('line',  RAILWAY_COLOR,  None,          1.5, 0.75, 'Railway'),
            ('mark',  STATION_COLOR,  STATION_EDGE,  STATION_MARKER, 'Railway Station'),
            ('mark',  PORT_COLOR,     PORT_EDGE,     PORT_MARKER,    'Port'),
            ('mark',  AIRPORT_COLOR,  AIRPORT_EDGE,  AIRPORT_MARKER, 'Aviation'),
        ]
        n_leg = len(leg_items)
        leg_h2 = title_h2 + n_leg * row_sp2 + map_data_h2 * 0.006
        leg_y0_2 = VY0 + map_data_h2 * 0.03

        bg2 = mpatches.FancyBboxPatch(
            (leg2_x0, leg_y0_2), leg2_w, leg_h2,
            boxstyle='square,pad=0', fc='#f2f2f2', ec='#aaa',
            lw=0.8, alpha=0.92, zorder=14, transform=ax.transData)
        ax.add_patch(bg2)
        ax.text(leg2_x0 + leg2_w * 0.5, leg_y0_2 + leg_h2 - map_data_h2 * 0.004,
                'Transport Infrastructure',
                ha='center', va='top', fontsize=18, fontfamily='Arial',
                color='#222', transform=ax.transData, zorder=15)

        for i, item in enumerate(leg_items):
            ry = leg_y0_2 + leg_h2 - title_h2 - (i + 0.5) * row_sp2
            if item[0] == 'line':
                _, color, _, lw, alpha, label = item
                ax.plot([sym_x - leg2_w*0.06, sym_x + leg2_w*0.06], [ry, ry],
                        color=color, linewidth=lw*1.5, alpha=alpha,
                        transform=ax.transData, zorder=15)
            elif item[0] == 'mark':
                _, color, edge, marker, label = item
                ax.scatter(sym_x, ry, s=80, c=color, marker=marker,
                           edgecolors=edge if edge else 'none', linewidths=0.3,
                           zorder=15, alpha=0.9)
            ax.text(text_x, ry, label,
                    ha='left', va='center', fontsize=16, fontfamily='Arial',
                    color='#333', transform=ax.transData, zorder=15)

    else:
        growth_items = [
            (petal_colors['total_road_km'],    'Road',            False),
            (petal_colors['paved_highway_km'], 'Paved Highway',   False),
            (petal_colors['railway_length_km'],'Railway',         False),
            (petal_colors[STATION_COL],        'Railway Station', False),
            (petal_colors['port_area_km2'],    'Port',            False),
            (petal_colors['airport_area_km2'], 'Aviation',         False),
            (None,                             'Negative Value',  True),
        ]
        n_growth = len(growth_items)
        leg2_h2 = title_h2 + n_growth * row_sp2 + map_data_h2 * 0.006
        leg2_y0_2 = VY0 + map_data_h2 * 0.03

        bg_growth = mpatches.FancyBboxPatch(
            (leg2_x0, leg2_y0_2), leg2_w, leg2_h2,
            boxstyle='square,pad=0', fc='#f2f2f2', ec='#aaa',
            lw=0.8, alpha=0.92, zorder=14, transform=ax.transData)
        ax.add_patch(bg_growth)
        ax.text(leg2_x0 + leg2_w * 0.5, leg2_y0_2 + leg2_h2 - map_data_h2 * 0.004,
                'Growth Rate',
                ha='center', va='top', fontsize=18, fontfamily='Arial',
                color='#222', transform=ax.transData, zorder=15)

        for i, (color, label, is_neg) in enumerate(growth_items):
            ry = leg2_y0_2 + leg2_h2 - title_h2 - (i + 0.5) * row_sp2
            rw = leg2_w * 0.10
            rh = row_sp2 * 0.6
            rx = sym_x - leg2_w * 0.05
            ry0 = ry - row_sp2 * 0.3
            if is_neg:
                rect = mpatches.FancyBboxPatch(
                    (rx, ry0), rw, rh,
                    boxstyle='square,pad=0', fc='#888888', ec='none',
                    alpha=0.45, zorder=15, transform=ax.transData)
                ax.add_patch(rect)
                for k in range(3):
                    ly = ry0 + rh * (k + 0.5) / 3
                    ax.plot([rx, rx + rw], [ly, ly],
                            color='white', lw=1, linestyle='solid',
                            alpha=0.85, transform=ax.transData, zorder=16)
            else:
                rect = mpatches.FancyBboxPatch(
                    (rx, ry0), rw, rh,
                    boxstyle='square,pad=0', fc=color, ec='none',
                    alpha=0.82, zorder=15, transform=ax.transData)
                ax.add_patch(rect)
            ax.text(text_x, ry, label,
                    ha='left', va='center', fontsize=16, fontfamily='Arial',
                    color='#333', transform=ax.transData, zorder=15)

        infra_items = [
            ('line',  ROAD_COLOR,     None,          1.5, 0.65, 'Road (+)'),
            ('line',  HIGHWAY_COLOR,  None,          2.0, 0.75, 'Paved Highway (+)'),
            ('line',  RAILWAY_COLOR,  None,          1.5, 0.75, 'Railway (+)'),
            ('mark',  STATION_COLOR,  STATION_EDGE,  STATION_MARKER, 'Railway Station (+)'),
            ('mark',  PORT_COLOR,     PORT_EDGE,     PORT_MARKER,    'Port (+)'),
            ('mark',  AIRPORT_COLOR,  AIRPORT_EDGE,  AIRPORT_MARKER, 'Aviation (+)'),
            ('patch', UNCHANGED_COLOR, None,         None, None, 'Existing in 2015'),
        ]
        n_infra = len(infra_items)
        leg1_h = title_h2 + n_infra * row_sp2 + map_data_h2 * 0.006
        leg1_y0 = leg2_y0_2 + leg2_h2 + leg_gap

        bg_infra = mpatches.FancyBboxPatch(
            (leg2_x0, leg1_y0), leg2_w, leg1_h,
            boxstyle='square,pad=0', fc='#f2f2f2', ec='#aaa',
            lw=0.8, alpha=0.92, zorder=14, transform=ax.transData)
        ax.add_patch(bg_infra)
        ax.text(leg2_x0 + leg2_w * 0.5, leg1_y0 + leg1_h - map_data_h2 * 0.004,
                'Transport Infrastructure',
                ha='center', va='top', fontsize=18, fontfamily='Arial',
                color='#222', transform=ax.transData, zorder=15)

        for i, item in enumerate(infra_items):
            ry = leg1_y0 + leg1_h - title_h2 - (i + 0.5) * row_sp2
            if item[0] == 'line':
                _, color, _, lw, alpha, label = item
                ax.plot([sym_x - leg2_w*0.06, sym_x + leg2_w*0.06], [ry, ry],
                        color=color, linewidth=lw*1.5, alpha=alpha,
                        transform=ax.transData, zorder=15)
            elif item[0] == 'mark':
                _, color, edge, marker, label = item
                ax.scatter(sym_x, ry, s=100, c=color, marker=marker,
                           edgecolors=edge if edge else 'none', linewidths=0.3,
                           zorder=15, alpha=0.9)
            elif item[0] == 'patch':
                _, color, _, _, _, label = item
                rect = mpatches.FancyBboxPatch(
                    (sym_x - leg2_w*0.06, ry - row_sp2*0.3),
                    leg2_w*0.12, row_sp2*0.6,
                    boxstyle='square,pad=0', fc=color, ec='none',
                    alpha=0.75, zorder=15, transform=ax.transData)
                ax.add_patch(rect)
            ax.text(text_x, ry, item[-1],
                    ha='left', va='center', fontsize=16, fontfamily='Arial',
                    color='#333', transform=ax.transData, zorder=15)

plt.show()
print("✓ Row 2 transport infrastructure distribution maps completed (six types)")


In [ ]:
# Cell | GDP and transport-growth time-series panels
# ============================================================================
from scipy import stats
import matplotlib.ticker as ticker

subplot_labels_row3 = ['G', 'H', 'I']

years_all = list(range(2015, 2025))  # 2015-2024

gdp_base_country = df_gdp[df_gdp['year'] == 2015][
    ['country', 'GDP_constant_2000_USD']].set_index('country')

gdp_growth_ts = {}
for yr in years_all:
    df_yr = df_gdp[df_gdp['year'] == yr][
        ['country', 'GDP_constant_2000_USD']].set_index('country')
    base = gdp_base_country['GDP_constant_2000_USD']
    yr_v = df_yr['GDP_constant_2000_USD']
    growth = (yr_v - base) / base.replace(0, np.nan) * 100
    gdp_growth_ts[yr] = growth.dropna()

trans_base_country = df_transport[df_transport['year'] == 2015][
    ['country'] + transport_6cols].set_index('country')

trans_growth_ts = {col: {} for col in transport_6cols}
for yr in years_all:
    df_yr = df_transport[df_transport['year'] == yr][
        ['country'] + transport_6cols].set_index('country')
    for col in transport_6cols:
        base_v = trans_base_country[col]
        yr_v   = df_yr[col]
        pct    = (yr_v - base_v) / base_v.replace(0, np.nan) * 100
        growth = pct.where(base_v != 0, np.nan)
        trans_growth_ts[col][yr] = growth.dropna()

def mean_sd(series_dict, years):
    means, lo, hi = [], [], []
    for yr in years:
        vals = series_dict[yr].values
        vals = vals[~np.isnan(vals)]
        n = len(vals)
        if n < 2:
            means.append(np.nan); lo.append(np.nan); hi.append(np.nan)
            continue
        m = np.mean(vals)
        sd = np.std(vals, ddof=1) 
        means.append(m)
        lo.append(m - sd)
        hi.append(m + sd)
    return np.array(means), np.array(lo), np.array(hi)

frames_all = []
for yr in years_all:
    gdp_s = gdp_growth_ts[yr].rename('gdp')
    trans_frames = {col: trans_growth_ts[col][yr] for col in transport_6cols}
    df_yr = pd.DataFrame({'gdp': gdp_s, **trans_frames})
    df_yr['year'] = yr
    frames_all.append(df_yr)
df_all = pd.concat(frames_all, ignore_index=True)
df_all = df_all[df_all['year'] != 2015]

df_pos = df_all[df_all['gdp'] > 0]
df_neg = df_all[df_all['gdp'] <= 0]

r_pos_all, r_neg_all = {}, {}
print("── Panel I: grouped Pearson correlations for pooled data ──")
print(f"  GDP>0 group: n={len(df_pos)},  GDP≤0 group: n={len(df_neg)}")
for col in transport_6cols:
    rp, pp = stats.pearsonr(df_pos[df_pos[col].notna()][col], df_pos[df_pos[col].notna()]['gdp'])
    rn, pn = stats.pearsonr(df_neg[df_neg[col].notna()][col], df_neg[df_neg[col].notna()]['gdp'])
    r_pos_all[col] = rp
    r_neg_all[col] = rn
    print(f"  {col}: r+={rp:.2f}(p={pp:.2f}), r-={rn:.2f}(p={pn:.2f})")

gdp_sum_2015 = df_gdp[df_gdp['year'] == 2015]['GDP_constant_2000_USD'].sum()
trans_sum_2015 = df_transport[df_transport['year'] == 2015][transport_6cols].sum()

sum_rows = []
for yr in years_all:
    gdp_s = df_gdp[df_gdp['year'] == yr]['GDP_constant_2000_USD'].sum()
    row = {'year': yr,
           'gdp_sum_growth': (gdp_s - gdp_sum_2015) / gdp_sum_2015 * 100}
    yr_sum = df_transport[df_transport['year'] == yr][transport_6cols].sum()
    for col in transport_6cols:
        base = trans_sum_2015[col]
        row[col + '_sum_growth'] = (yr_sum[col] - base) / base * 100 if base != 0 else yr_sum[col]
    sum_rows.append(row)
df_sum = pd.DataFrame(sum_rows)
df_sum = df_sum[df_sum['year'] != 2015]

r_sum_all = {}
print("── Panel I: Pearson correlations for continent-wide aggregate growth rates ──")
for col in transport_6cols:
    r_s, p_s = stats.pearsonr(df_sum[col + '_sum_growth'], df_sum['gdp_sum_growth'])
    r_sum_all[col] = r_s
    print(f"  {col}: r={r_s:.2f}(p={p_s:.2f})")

transport_6labels_short_base = {
    'total_road_km':    'Road',
    'paved_highway_km': 'Paved\nHighway',
    'railway_length_km':'Railway',
    'port_area_km2':    'Port',
    'airport_area_km2': 'Aviation',
}
transport_6labels_short = transport_6labels_short_base.copy()
transport_6labels_short[STATION_COL] = 'Railway\nStation'

GDP_COLOR = '#2166ac'
SUM_COLOR = '#333333'

FIG_W3 = 26
FIG_H3 = 7.5

fig3 = plt.figure(figsize=(FIG_W3, FIG_H3))
fig3.patch.set_facecolor('white')

gap_gh     = 0.04
gap_hi     = 0.04

fig_left   = 0.06
fig_right  = 0.98
fig_bottom = 0.13
fig_top    = 0.97

total_w   = fig_right - fig_left - gap_gh - gap_hi
subplot_w = total_w / 3
fig_h     = fig_top - fig_bottom

ax_g_left = fig_left
ax_h_left = fig_left + subplot_w + gap_gh
ax_i_left = fig_left + 2 * subplot_w + gap_gh + gap_hi

ax_g = fig3.add_axes([ax_g_left, fig_bottom, subplot_w, fig_h])
ax_h = fig3.add_axes([ax_h_left, fig_bottom, subplot_w, fig_h])
ax_i = fig3.add_axes([ax_i_left, fig_bottom, subplot_w, fig_h])
axes3 = [ax_g, ax_h, ax_i]

TICK_FS  = 18
LABEL_FS = 18
LEGEND_FS = 16
ALPHA_CI = 0.18
DASH_LW  = 4.0
DASH_HALF = 0.22
CONN_LW  = 1.8

ax_g = axes3[0]
gdp_means, gdp_lo, gdp_hi = mean_sd(gdp_growth_ts, years_all)

ax_g.fill_between(years_all, gdp_lo, gdp_hi, color=GDP_COLOR, alpha=ALPHA_CI)
ax_g.plot(years_all, gdp_means, color=GDP_COLOR, lw=2.2, marker='o',
          markersize=5, markerfacecolor='white', markeredgewidth=1.5, zorder=5)
ax_g.axhline(0, color='#aaa', lw=0.8, linestyle='--', zorder=1)

ax_g.set_xlim(2014.5, 2024.5)
ax_g.set_xticks(years_all)
ax_g.set_xticklabels([str(y) for y in years_all], rotation=45, ha='right',
                     rotation_mode='anchor', fontsize=TICK_FS)
ax_g.tick_params(axis='y', labelsize=TICK_FS)
ax_g.set_ylabel('GDP Growth Rate vs 2015 (%)', fontsize=LABEL_FS, fontfamily='Arial')
ax_g.spines[['top', 'right']].set_visible(False)
ax_g.spines[['left', 'bottom']].set_linewidth(0.8)
ax_g.grid(axis='y', color='#ddd', lw=0.6, zorder=0)
ax_g.text(-0.12, 1.04, subplot_labels_row3[0],
          transform=ax_g.transAxes, fontsize=35, fontweight='bold',
          va='top', fontfamily='Arial', color='#111')

ax_h = axes3[1]

lines_h = []

# Estimate the model for each transport indicator.
for col in transport_6cols:
    color = petal_colors[col]
    label = transport_6labels_vis[col]
    
    t_means, t_lo, t_hi = mean_sd(trans_growth_ts[col], years_all)
    
    ax_h.fill_between(years_all, t_lo, t_hi, color=color, alpha=ALPHA_CI)
    
    line_style = '-'
    marker_style = 'o'
        
    ln, = ax_h.plot(years_all, t_means, color=color, lw=2.2, marker=marker_style,
                    markersize=5, markerfacecolor='white', markeredgewidth=1.5,
                    label=label, zorder=5, linestyle=line_style)
    lines_h.append(ln)

ax_h.axhline(0, color='#aaa', lw=0.8, linestyle='-', zorder=1)

# =========================================================================
ax_h.set_yscale('symlog', base=10, linthresh=10.0)

exponents_pos = np.arange(1, 8) 
ticks = [0, -10, -100]
for exp in exponents_pos:
    ticks.append(10 ** exp)
ticks = sorted(list(set(ticks)))
ax_h.yaxis.set_major_locator(ticker.FixedLocator(ticks))

def format_func(value, tick_number):
    if value == 0:
        return '0'
    sign = '-' if value < 0 else ''
    exp = int(round(np.log10(abs(value))))
    if exp == 1:
        return f'${sign}10$'
    return f'${sign}10^{{{exp}}}$'

limit_upper = 10 ** 7.2 
ax_h.set_ylim(-100, limit_upper)

ax_h.yaxis.set_major_formatter(ticker.FuncFormatter(format_func))
# =========================================================================

ax_h.set_xlim(2014.5, 2024.5)
ax_h.set_xticks(years_all)
ax_h.set_xticklabels([str(y) for y in years_all], rotation=45, ha='right',
                     rotation_mode='anchor', fontsize=TICK_FS)
ax_h.tick_params(axis='y', labelsize=TICK_FS)
ax_h.set_ylabel('Transport Growth Rate vs 2015 (%)', fontsize=LABEL_FS, fontfamily='Arial')

ax_h.spines[['top', 'right']].set_visible(False)
ax_h.spines[['left', 'bottom']].set_linewidth(0.8)

ax_h.grid(axis='y', color='#ddd', lw=0.6, zorder=0, alpha=0.8)

labels_h = [ln.get_label() for ln in lines_h]
ax_h.legend(lines_h, labels_h, fontsize=LEGEND_FS, framealpha=0.8, edgecolor='#ccc',
            loc='upper left', prop={'family': 'Arial', 'size': LEGEND_FS})

ax_h.text(-0.12, 1.04, subplot_labels_row3[1],
          transform=ax_h.transAxes, fontsize=35, fontweight='bold',
          va='top', fontfamily='Arial', color='#111')

ax_i = axes3[2]

x_pos = np.arange(len(transport_6cols))
x_labels_short = [transport_6labels_short[c] for c in transport_6cols]

# Estimate the model for each transport indicator.
for xi, col in enumerate(transport_6cols):
    color = petal_colors[col]
    rp = r_pos_all[col]
    rn = r_neg_all[col]

    ax_i.plot([xi, xi], [rn, rp], color=color, lw=CONN_LW, alpha=0.75, zorder=4)

    ax_i.plot([xi - DASH_HALF, xi + DASH_HALF], [rp, rp],
              color=color, lw=DASH_LW, solid_capstyle='round', zorder=5)
    ax_i.plot([xi - DASH_HALF, xi + DASH_HALF], [rn, rn],
              color=color, lw=DASH_LW, linestyle='--',
              solid_capstyle='round', zorder=5)

    ax_i.text(xi + DASH_HALF + 0.05, rp, f'{rp:.2f}',
              va='center', ha='left', fontsize=TICK_FS - 2,
              fontfamily='Arial', color=color)
    ax_i.text(xi + DASH_HALF + 0.05, rn, f'{rn:.2f}',
                  va='center', ha='left', fontsize=TICK_FS - 2,
                  fontfamily='Arial', color=color)

r_sum_vals = [r_sum_all[col] for col in transport_6cols]
ax_i.plot(x_pos, r_sum_vals, color=SUM_COLOR, lw=2.2, marker='D',
          markersize=7, markerfacecolor='white', markeredgewidth=2,
          linestyle='-', zorder=6, label='Continental')
for xi, r_s in enumerate(r_sum_vals):
    col = transport_6cols[xi]
    if col == 'port_area_km2':
        ax_i.text(xi - 0.4 , r_s, f'{r_s:.2f}',
                  va='center', ha='center', fontsize=TICK_FS - 2,
                  fontfamily='Arial', color=SUM_COLOR)
    else:
        ax_i.text(xi, r_s + 0.04, f'{r_s:.2f}',
                  va='bottom', ha='center', fontsize=TICK_FS - 2,
                  fontfamily='Arial', color=SUM_COLOR)

ax_i.plot([], [], color='#666', lw=DASH_LW, linestyle='-',
          label='GDP growth rate > 0 group')
ax_i.plot([], [], color='#666', lw=DASH_LW, linestyle='--',
          label='GDP growth rate ≤ 0 group')

ax_i.axhline(0, color='#aaa', lw=0.8, linestyle='--', zorder=1)
ax_i.set_xlim(-0.6, len(transport_6cols) - 0.3)
ax_i.set_ylim(-1, 1)
ax_i.set_xticks(x_pos)
ax_i.set_xticklabels(x_labels_short, rotation=0, fontsize=TICK_FS-2, fontfamily='Arial')
ax_i.tick_params(axis='y', labelsize=TICK_FS)
ax_i.set_ylabel('Pearson r', fontsize=LABEL_FS, fontfamily='Arial')
ax_i.spines[['top', 'right']].set_visible(False)
ax_i.spines[['left', 'bottom']].set_linewidth(0.8)
ax_i.grid(axis='y', color='#ddd', lw=0.6, zorder=0)
ax_i.legend(fontsize=LEGEND_FS, framealpha=0.9, edgecolor='#ccc', loc='best',
           prop={'family': 'Arial', 'size': LEGEND_FS})
ax_i.text(-0.12, 1.04, subplot_labels_row3[2],
          transform=ax_i.transAxes, fontsize=35, fontweight='bold',
          va='top', fontfamily='Arial', color='#111')

plt.show()
print("✓ Row 3 time-series plots completed (six types)")
